---
title: "Comparing Solutions"
image: image.png
toc: true
execute: 
  enabled: true
---

In [42]:
from lokigi.site import SiteProblem

Let's first load up a problem. 

We'll just load in the information that needs to be shared. 

In [43]:
problem = SiteProblem()

problem.add_sites(
    "../../../sample_data/devon_mius.geojson",
    candidate_id_col="Facility_Name"
    )

problem.add_region_geometry_layer(
    "https://github.com/hsma-programme/h6_3c_interactive_plots_travel/raw/main/h6_3c_interactive_plots_travel/example_code/LSOA_2011_Boundaries_Super_Generalised_Clipped_BSC_EW_V4.geojson",
    common_col="LSOA11NM"
    )

We'll then take a copy of our problem and add our car travel matrix.

In [44]:
problem_car = problem.copy()

problem_car.add_travel_matrix(
    travel_matrix_df="../../../sample_data/devon_miu_travel_matrix.csv",
    source_col="from_id",
    unit="minutes",
    )

Then we can repeat, adding our public transport matrix. 

In [45]:
problem_public_transport = problem.copy()

problem_public_transport.add_travel_matrix(
    travel_matrix_df="../../../sample_data/devon_miu_travel_matrix_public_transport_extended.csv",
    source_col="from_id",
    unit="minutes",
    )

We'll solve each. 

In [46]:
solution_car = problem_car.solve(p=12, threshold_for_coverage=20)
solution_public_transport = problem_public_transport.solve(p=12, threshold_for_coverage=60)

C:\lokigi\lokigi\site.py:1414: UserWarning: No demand data was provided. Demand from all regions has been assumed to be equal.If you wish to override this, run .add_demand() to add your site dataframe before running .solve() again.You can use the .show_demand_format() to see the expected format beforehand.
  warn(


  0%|          | 0/455 [00:00<?, ?it/s]

C:\lokigi\lokigi\site.py:1414: UserWarning: No demand data was provided. Demand from all regions has been assumed to be equal.If you wish to override this, run .add_demand() to add your site dataframe before running .solve() again.You can use the .show_demand_format() to see the expected format beforehand.
  warn(


  0%|          | 0/455 [00:00<?, ?it/s]

In [47]:
solution_car.show_solutions().head(2)

,site_names,site_indices,coverage_threshold,weighted_average,unweighted_average,90th_percentile,max,proportion_within_coverage_threshold,problem_df
0,"[North Devon District Hospital, Honiton Hospit...","[0, 1, 2, 3, 5, 6, 7, 8, 9, 11, 12, 14]",20,15.08,15.08,26.0,54.0,0.75,from_id from_id_x North D...
1,"[North Devon District Hospital, Honiton Hospit...","[0, 1, 2, 3, 4, 5, 6, 7, 8, 11, 12, 14]",20,15.10,15.10,26.0,54.0,0.75,from_id from_id_x North D...


In [48]:
solution_public_transport.show_solutions().head(2)

,site_names,site_indices,coverage_threshold,weighted_average,unweighted_average,90th_percentile,max,proportion_within_coverage_threshold,problem_df
0,"[North Devon District Hospital, Honiton Hospit...","[0, 1, 2, 3, 5, 6, 7, 8, 10, 11, 12, 14]",60,58.63,58.63,99.0,262.0,0.6,from_id from_id_x North D...
1,"[North Devon District Hospital, Honiton Hospit...","[0, 1, 2, 3, 4, 5, 6, 7, 8, 10, 11, 12]",60,58.66,58.66,99.0,262.0,0.6,from_id from_id_x North D...


We could pull out the best solution for car from our public transport example. 

In [49]:
best_indices_car = solution_car.return_best_combination_site_indices()

public_transport_solution_df = solution_public_transport.show_solutions()


public_transport_solution_df[
    public_transport_solution_df["site_indices"].apply(set) == set(best_indices_car)
    ]

,site_names,site_indices,coverage_threshold,weighted_average,unweighted_average,90th_percentile,max,proportion_within_coverage_threshold,problem_df
46,"[North Devon District Hospital, Honiton Hospit...","[0, 1, 2, 3, 5, 6, 7, 8, 9, 11, 12, 14]",60,59.61,59.61,98.4,262.0,0.58,from_id from_id_x North D...


So our best solution for car is our 47th best solution for public transport! 

In [50]:
best_indices_public_transport = solution_public_transport.return_best_combination_site_indices()

car_solution_df = solution_car.show_solutions()

car_solution_df[
    car_solution_df["site_indices"].apply(set) == set(best_indices_public_transport)
    ]

,site_names,site_indices,coverage_threshold,weighted_average,unweighted_average,90th_percentile,max,proportion_within_coverage_threshold,problem_df
2,"[North Devon District Hospital, Honiton Hospit...","[0, 1, 2, 3, 5, 6, 7, 8, 10, 11, 12, 14]",20,15.11,15.11,26.0,54.0,0.74,from_id from_id_x North D...


But our best public transport solution is our 2nd best car travel solution.